# Embeddings Pre-entrenados y Limpieza de Texto con spaCy

### De entrenar nuestros propios embeddings a reusar unos ya hechos

En el notebook anterior, la capa `Embedding` de nuestro clasificador aprendia sus vectores desde cero, usando solamente las review de entrenamiento. Funciona, pero tiene un costo: necesitamos muchos datos propios para que esos vectores terminen siendo buenos.

Hoy vamos a hacer lo contrario: usar vectores que **ya fueron entrenados por alguien mas**, sobre una cantidad enorme de texto (mucho mas de lo que nosotros podriamos procesar en una clase). En vez de entrenar embeddings propios, reusamos unos pre-entrenados y aprovechamos todo ese conocimiento previo.

Para esto vamos a usar **spaCy**, una libreria muy usada en la industria para procesamiento de lenguaje natural, que trae modelos con vectores pre-entrenados listos para usar.

### El dataset de esta semana

Vamos a trabajar sobre un dataset real: **tweets de clientes sobre aerolineas de Estados Unidos**, clasificados originalmente como positivos, negativos o neutrales, con la razon del reclamo cuando es negativo (vuelo demorado, mal servicio, equipaje perdido, etc). Es un dataset clasico para practicar analisis de sentimiento sobre feedback de clientes real.

En este notebook nos enfocamos en dos cosas:

1. Entender que hace spaCy cuando procesamos un texto, y como usar sus vectores pre-entrenados.
2. Limpiar el dataset (texto crudo, con menciones y links) para dejarlo listo para el resto de la semana.

## 0. Preparando el entorno

spaCy no viene con los modelos de lenguaje instalados por defecto: hay que descargarlos aparte. Vamos a usar `en_core_web_lg`, la version "large" del modelo en ingles (la version "large" es la que trae vectores de palabras reales; las versiones "small" no los traen).

Mas info sobre el model de spacy:  
https://spacy.io/models/en

In [1]:
!pip install spacy
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re # "regular expressions": libreria buscar, extraer, reemplazar y validar patrones de texto

import pandas as pd
import matplotlib.pyplot as plt
import spacy

## 1. Cargando el modelo pre-entrenado

Cargar el modelo es una sola linea. Adentro de ese objeto `nlp` ya estan todos los vectores pre-entrenados, listos para usar.
<br> <br>
"en_core_web_lg" es un modelo de NLP de gran tamaño para el idioma inglés. Tambien existe para otra lengua (por ejemplo "es_core_news_lg" para espanol)

In [3]:
nlp = spacy.load("en_core_web_lg")

## 2. Que hace spaCy cuando procesamos un texto

Cuando le pasamos un texto al objeto `nlp`, spaCy lo tokeniza y ademas calcula, para cada token, un monton de informacion linguistica: su lema (la forma "base" de la palabra), su categoria gramatical, y si es una stopword (una palabra muy frecuente y poco informativa, como "the" o "is").

Probemos con una oracion de ejemplo.

In [4]:
texto_ejemplo = "The flight was delayed and the crew was extremely rude to us."
doc_ejemplo = nlp(texto_ejemplo)

In [5]:
doc_ejemplo[2].pos_

doc_ejemplo[2].lemma_

'be'

In [6]:
for token in doc_ejemplo:
    print(f"{token.text:<12} lema={token.lemma_:<12} categoria={token.pos_:<8} es_stopword={token.is_stop}")

The          lema=the          categoria=DET      es_stopword=True
flight       lema=flight       categoria=NOUN     es_stopword=False
was          lema=be           categoria=AUX      es_stopword=True
delayed      lema=delay        categoria=VERB     es_stopword=False
and          lema=and          categoria=CCONJ    es_stopword=True
the          lema=the          categoria=DET      es_stopword=True
crew         lema=crew         categoria=NOUN     es_stopword=False
was          lema=be           categoria=AUX      es_stopword=True
extremely    lema=extremely    categoria=ADV      es_stopword=False
rude         lema=rude         categoria=ADJ      es_stopword=False
to           lema=to           categoria=ADP      es_stopword=True
us           lema=we           categoria=PRON     es_stopword=True
.            lema=.            categoria=PUNCT    es_stopword=False


Fijate que "was" se lematiza como "be", y que palabras como "the", "was" o "and" quedan marcadas como stopwords. Vamos a usar exactamente esta informacion mas adelante para limpiar el dataset.

ejemplo de lema en espanol :  
"corriendo, corrió, correremos" ---> "correr"

## 3. Los vectores pre-entrenados

Cada token (y cada documento completo) tiene un vector asociado, calculado a partir de esos embeddings pre-entrenados.

In [7]:
vector_palabra = nlp("flight")[0].vector

print("Cantidad de dimensiones del vector:", vector_palabra.shape)
print(vector_palabra[:10], "...")

Cantidad de dimensiones del vector: (300,)
[ 0.32473  -0.46939   0.16104   0.1247    0.483     0.40204  -0.18667
  0.54197   0.025085  2.5308  ] ...


Lo interesante de estos vectores es que podemos medir que tan parecidos son dos textos, calculando la similitud (coseno) entre sus vectores. spaCy nos da esto directo con el metodo `.similarity()`.

Probemos comparando una queja sobre vuelos demorados contra otra queja parecida, y despues contra un comentario positivo que no tiene nada que ver.

In [8]:
comentario_1 = nlp("My flight was delayed for three hours.")
comentario_2 = nlp("The plane took off two hours late.")
comentario_3 = nlp("The flight attendants were amazing and so kind.")

print("Similitud entre comentario_1 y comentario_2:", comentario_1.similarity(comentario_2))
print()
print("Similitud entre comentario_1 y comentario_3:", comentario_1.similarity(comentario_3))

Similitud entre comentario_1 y comentario_2: 0.9156274199485779

Similitud entre comentario_1 y comentario_3: 0.8532575964927673


Los dos primeros comentarios hablan de lo mismo (un vuelo demorado) con palabras totalmente distintas, y aun asi el modelo detecta que son parecidos. Esa es la gracia de trabajar con embeddings en vez de comparar palabras exactas. Vamos a volver a usar esta idea de similitud en el proximo notebook, para buscar los comentarios mas parecidos a uno dado.

## 4. Un paso mas alla: embeddings contextuales (BERT)

Los vectores que acabamos de usar tienen una limitacion importante: son **estaticos**. Cada palabra tiene un unico vector, fijo, sin importar en que oracion aparezca. Vamos a comprobarlo con la palabra "right", que en ingles puede significar "derecha" (una direccion) o "correcto / tener razon" -- dos significados completamente distintos segun el contexto.

In [9]:
import numpy as np

# creamos frase con diferente contexto por el palabra "right"
oracion_a = nlp("Turn right at the next corner.")
oracion_b = nlp("You are completely right about this.")

# Buscamos la palabra "right" en cada oracion y obtenemos su vector
vector_a = oracion_a[1].vector
vector_b = oracion_b[3].vector

# Comparamos los dos vectores elemento por elemento
print("¿Son exactamente el mismo vector?", np.array_equal(vector_a, vector_b))

¿Son exactamente el mismo vector? True


Como era de esperar, spaCy le asigna a "right" exactamente el mismo vector en las dos oraciones, aunque el significado sea distinto en cada caso. El modelo no tiene forma de distinguir estos dos usos: solo sabe que la palabra es "right", y siempre le devuelve la misma fila de su tabla de vectores.

Esta limitacion motivo el desarrollo de los **embeddings contextuales**: modelos donde el vector de una palabra se calcula en base a las palabras que la rodean, de forma que la misma palabra puede tener vectores distintos segun el contexto en el que aparece.  
**BERT** (Bidirectional Encoder Representations from Transformers) es uno de estos modelos.

Carguemos BERT con la libreria `transformers` de Hugging Face y repitamos la comparacion.

--> BERT es un Transformer-based language model !

In [10]:
!pip install transformers torch

In [11]:
import torch
from transformers import AutoTokenizer, AutoModel

# Cargamos el tokenizador de BERT para convertir texto en tokens/IDs
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Cargamos el modelo BERT preentrenado
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Ponemos BERT en modo evaluacion: no estamos entrenandolo ni modificando sus pesos
bert_model.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

A diferencia de spaCy, BERT no le asigna un vector a cada palabra de forma aislada: primero tokeniza la oracion completa, y despues calcula un vector por token teniendo en cuenta toda la oracion.
Armamos una funcion chica que, dada una oracion y una palabra, nos devuelva el vector contextual de esa palabra especifica.

In [12]:
def vector_contextual(oracion, palabra):  # Obtiene el vector contextual de una palabra dentro de una oracion

    encoding = bert_tokenizer(oracion, return_tensors="pt")  # Convertimos la oracion en los tokens que BERT entiende y los guardamos como tensores de PyTorch

    with torch.no_grad():  # No calculamos gradientes porque solo estamos usando BERT para obtener los vectores
        salida = bert_model(**encoding)  # Pasamos los tokens por BERT y obtenemos sus representaciones

    tokens = bert_tokenizer.convert_ids_to_tokens(encoding["input_ids"][0]) # Convertimos los IDs generados por el tokenizador de nuevo a los tokens de BERT

    indice = tokens.index(palabra)  # Buscamos la posicion de la palabra que nos interesa

    return salida.last_hidden_state[0, indice]  # Devolvemos el vector contextual de esa palabra

In [13]:
vector_bert_a = vector_contextual("Turn right at the next corner.", "right")
vector_bert_b = vector_contextual("You are completely right about this.", "right")

similitud_bert = torch.nn.functional.cosine_similarity(vector_bert_a, vector_bert_b, dim=0)
print("Similitud contextual (BERT) para 'right':", similitud_bert.item())

Similitud contextual (BERT) para 'right': 0.30060073733329773


Esta vez la similitud es menor a 1: BERT calculo dos vectores distintos para "right", adaptados a cada contexto, en vez de devolver siempre la misma fila de una tabla como hace spaCy.

Esta es la idea central detras de los embeddings contextuales: ningun token tiene un significado fijo, sino uno que depende de todo lo que lo rodea en la oracion. Es tambien la razon por la que estos modelos son mas lentos y pesados que spaCy: en vez de una simple consulta a una tabla, hay que correr una red neuronal completa cada vez que procesamos un texto.

Para el resto de este taller nos alcanza con los vectores estaticos de spaCy, mucho mas livianos. Pero vale la pena saber que esta alternativa existe, sobre todo para tareas donde el contexto especifico de cada palabra importa mucho.

## 5. El dataset: tweets de clientes sobre aerolineas

Es el dataset **Twitter US Airline Sentiment**: mas de 14,000 tweets reales dirigidos a aerolineas de Estados Unidos, recolectados en 2015. Cada tweet ya viene etiquetado por personas como positivo, negativo o neutral, y los negativos tienen ademas una razon (vuelo demorado, mal servicio, equipaje perdido, etc). Vamos a guardar esa etiqueta original aparte, para mas adelante comparar contra lo que nosotros mismos calculemos.

Lo cargamos directamente desde el csv, sin necesidad de descargar nada a mano.

In [14]:
url_dataset = "https://raw.githubusercontent.com/Camille-Le-Roy/data_analytics_resources/main/Datasets/"
df = pd.read_csv(url_dataset + "Tweets_Airlines.csv")

columnas_utiles = ["tweet_id", "airline", "text", "airline_sentiment", "negativereason"]

df = pd.read_csv(url_dataset + "Tweets_Airlines.csv", usecols=columnas_utiles)

df = df.rename(columns={
    "tweet_id": "id",
    "airline": "aerolinea",
    "text": "texto",
    "airline_sentiment": "sentimiento_original",
    "negativereason": "razon_negativa",
})

print("Cantidad de tweets:", len(df))
df.head()

Cantidad de tweets: 14640


,id,sentimiento_original,razon_negativa,aerolinea,texto
0,570306133677760513,neutral,NaN,Virgin America,@VirginAmerica What @dhepburn said.
1,570301130888122368,positive,NaN,Virgin America,@VirginAmerica plus you've added commercials t...
2,570301083672813571,neutral,NaN,Virgin America,@VirginAmerica I didn't today... Must mean I n...
3,570301031407624196,negative,Bad Flight,Virgin America,@VirginAmerica it's really aggressive to blast...
4,570300817074462722,negative,Can't Tell,Virgin America,@VirginAmerica and it's a really big bad thing...


In [15]:
df["sentimiento_original"].value_counts()

,count
sentimiento_original,
negative,9178
neutral,3099
positive,2363


In [16]:
df["razon_negativa"].unique()

array([nan, 'Bad Flight', "Can't Tell", 'Late Flight',
       'Customer Service Issue', 'Flight Booking Problems',
       'Lost Luggage', 'Flight Attendant Complaints', 'Cancelled Flight',
       'Damaged Luggage', 'longlines'], dtype=object)

## 6. Limpiando el texto

Los tweets tienen varias cosas que no aportan mucho al significado: menciones a usuarios (`@VirginAmerica`), links, numeros, signos de puntuacion. Vamos a limpiar el texto en dos pasos, porque **la limpieza que conviene depende de para que la vamos a usar despues**:

- Para **analisis de sentimiento** (proximo notebook) conviene sacar las menciones y los links, pero **conservar el resto de la oracion tal cual**: cosas como "not" o "n't" cambian completamente el sentido de una frase, y si las sacamos como si fueran stopwords, el clasificador de sentimiento va a leer mal el texto.
- Para **modelado de topicos** (el ultimo notebook de la semana) si conviene una limpieza mas agresiva: pasar todo a minusculas, sacar stopwords y puntuacion, y quedarnos con el lema de cada palabra. Ahi el orden de las palabras no importa, así que no hay riesgo en "desordenar" la oracion.

Por eso vamos a generar dos columnas de texto limpio, no una sola.

### 6.1 Limpieza liviana: sacar menciones y links

Esto lo hacemos con la libreria "regular expression" (re), antes de pasar el texto por spaCy.

In [17]:
def quitar_menciones_y_links(texto):
    texto = re.sub(r"@\w+", "", texto)  # menciones tipo @VirginAmerica
    texto = re.sub(r"http\S+", "", texto)  # links
    return texto.strip()

# re.sub(pattern, "", texto) ---> Encuentra un patrón en el texto y reemplázalo con otra cosa


df["texto_sentimiento"] = df["texto"].apply(quitar_menciones_y_links)
df[["texto", "texto_sentimiento"]].head()

,texto,texto_sentimiento
0,@VirginAmerica What @dhepburn said.,What said.
1,@VirginAmerica plus you've added commercials t...,plus you've added commercials to the experienc...
2,@VirginAmerica I didn't today... Must mean I n...,I didn't today... Must mean I need to take ano...
3,@VirginAmerica it's really aggressive to blast...,"it's really aggressive to blast obnoxious ""ent..."
4,@VirginAmerica and it's a really big bad thing...,and it's a really big bad thing about it


### 6.2 Limpieza agresiva: minusculas, sin stopwords, lematizado

Aca es donde usamos spaCy: para cada tweet vamos a quedarnos solo con los lemas de las palabras que no sean stopwords ni puntuacion.

Como tenemos miles de tweets, en vez de llamar a `nlp(texto)` uno por uno, usamos `nlp.pipe(...)`, que procesa todos los textos en lote de forma mucho mas eficiente. Tambien desactivamos partes del modelo que no necesitamos para esta tarea (el parser de dependencias y el reconocedor de entidades), para que sea mas rapido.

In [18]:
test = nlp(df["texto_sentimiento"].iloc[5])
# se puede checkear si un token es stop word o punctuacion o otro
print(test)
test[5].is_stop

seriously would pay $30 a flight for seats that didn't have this playing.
it's really the only bad thing about flying VA


True

In [19]:
def limpiar_para_topicos(doc):

    lemas = [
        token.lemma_.lower()  # Convertimos el lema a minusculas

        for token in doc  # Recorremos todos los tokens del documento
        if not token.is_stop  # Ignoramos palabras muy comunes como "the", "a", "is", etc.
        and not token.is_punct  # Ignoramos signos de puntuacion
        and not token.is_space  # Ignoramos espacios y saltos de linea

    ]

    return " ".join(lemas)  # Unimos los lemas restantes en un solo texto

In [20]:
test = nlp(df["texto_sentimiento"].iloc[5])

print(test)
print()
print(limpiar_para_topicos(test))

seriously would pay $30 a flight for seats that didn't have this playing.
it's really the only bad thing about flying VA

seriously pay $ 30 flight seat playing bad thing fly va


In [21]:
textos_limpios = []  # Creamos una lista donde guardaremos los textos ya procesados

# Procesamos todos los textos con spaCy
for doc in nlp.pipe(
    df["texto_sentimiento"],
    disable=["parser", "ner"],  # No necesitamos parser ni NER, solo la tokenizacion/lemas
    batch_size=200  # Procesamos 200 textos a la vez para hacerlo mas eficiente
):

    textos_limpios.append(
        limpiar_para_topicos(doc)  # Limpiamos cada documento y guardamos el resultado
    )

df["texto_topicos"] = textos_limpios
df[["texto", "texto_sentimiento", "texto_topicos"]].head(10)

,texto,texto_sentimiento,texto_topicos
0,@VirginAmerica What @dhepburn said.,What said.,say
1,@VirginAmerica plus you've added commercials t...,plus you've added commercials to the experienc...,plus add commercial experience tacky
2,@VirginAmerica I didn't today... Must mean I n...,I didn't today... Must mean I need to take ano...,today mean need trip
3,@VirginAmerica it's really aggressive to blast...,"it's really aggressive to blast obnoxious ""ent...",aggressive blast obnoxious entertainment guest...
4,@VirginAmerica and it's a really big bad thing...,and it's a really big bad thing about it,big bad thing
5,@VirginAmerica seriously would pay $30 a fligh...,seriously would pay $30 a flight for seats tha...,seriously pay $ 30 flight seat playing bad thi...
6,"@VirginAmerica yes, nearly every time I fly VX...","yes, nearly every time I fly VX this “ear worm...",yes nearly time fly vx ear worm will away
7,@VirginAmerica Really missed a prime opportuni...,Really missed a prime opportunity for Men With...,miss prime opportunity man hats parody
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","Well, I didn't…but NOW I DO! :-D",didn't :-d
9,"@VirginAmerica it was amazing, and arrived an ...","it was amazing, and arrived an hour early. You...",amazing arrive hour early good


## 7. Guardando el dataset limpio

Este dataset (con las tres versiones del texto) es el punto de partida del resto de la semana: en el proximo notebook usamos `texto_sentimiento` para clasificar cada tweet, y mas adelante usamos `texto_topicos` para el modelado de topicos.

In [22]:
df.to_csv("feedback_aerolineas_limpio.csv", index=False)
print("Guardado. Filas:", len(df))

Guardado. Filas: 14640


## Resumen

- **Embeddings pre-entrenados**: en vez de entrenar vectores propios (como hicimos con `nn.Embedding` en los notebooks anteriores), reusamos vectores ya entrenados por spaCy sobre un corpus enorme.
- **Embeddings contextuales**: a diferencia de los vectores estaticos de spaCy, modelos como BERT calculan un vector distinto para cada palabra segun el contexto en el que aparece. Es una alternativa mas pesada, pero mas precisa cuando el significado depende mucho del contexto.
- **Similitud semantica**: gracias a esos vectores, podemos comparar el significado de dos textos aunque no compartan las mismas palabras.
- **Limpieza de texto**: no existe "una" limpieza correcta. Depende de la tarea: para analisis de sentimiento conviene tocar lo menos posible la oracion, para modelado de topicos conviene una limpieza mas agresiva (minusculas, sin stopwords, lematizado).

En el proximo notebook usamos este dataset ya limpio para clasificar cada tweet como positivo, neutral o negativo.